# sweets example 1 — Sentinel-1 burst-subset workflow

End-to-end InSAR run over a small AOI in Los Angeles (Long Beach / San Pedro) using the default `--source safe` path: raw Sentinel-1 bursts downloaded as just-the-bursts-that-intersect-the-AOI via `burst2safe`, then geocoded by COMPASS, then handed to dolphin for phase linking, interferogram network selection, unwrapping, and timeseries inversion.

**AOI**: small rectangle over LA, same polygon as the OPERA CSLC and NISAR example notebooks.

**Time range**: 2025-12-01 -> 2025-12-31 (a few Sentinel-1 repeat passes on descending track 71).

**Expected wall time**: ~5-10 minutes on a fast connection, dominated by SAFE download + COMPASS geocoding. Outputs land under `dolphin/` and the final velocity raster is `dolphin/timeseries/velocity.tif` in meters/year.

## Earthdata credentials

sweets downloads Sentinel-1 bursts, OPERA CSLCs, NISAR GSLCs and tropo data from NASA Earthdata endpoints. Every subsystem used below (`burst2safe`, `opera-utils`, `asf_search`) expects a `~/.netrc` entry like:

```
machine urs.earthdata.nasa.gov
  login YOUR_EARTHDATA_USERNAME
  password YOUR_EARTHDATA_PASSWORD
```

If you don't have credentials yet, register for free at <https://urs.earthdata.nasa.gov/users/new>.

## Configure the run

sweets takes a YAML config; the `sweets config` subcommand builds one from a few flags. `--track 71` pins the descending Sentinel-1 track that covers LA, and `--swaths IW2` restricts burst2safe to a single subswath (it refuses to straddle swaths in one SAFE).

In [ ]:
from pathlib import Path

WORK_DIR = Path("./example_s1_burst").resolve()
WORK_DIR.mkdir(exist_ok=True)
CONFIG_YAML = WORK_DIR / "sweets_config.yaml"
print(WORK_DIR)

In [ ]:
!sweets config \
  --bbox=-118.3957 33.7284 -118.3459 33.772 \
  --start 2025-12-01 --end 2025-12-31 \
  --track 71 \
  --swaths IW2 \
  --out-dir {WORK_DIR}/data \
  --work-dir {WORK_DIR} \
  --output {CONFIG_YAML}

Take a look at the generated config — this is the file that `sweets run` will execute. Feel free to edit any of the long-tail knobs (dolphin half-window, COMPASS posting, water-mask buffer, etc.) directly in the YAML.

In [ ]:
!head -60 {CONFIG_YAML}

## Run the workflow

Step 1 downloads the SAFEs (just the intersecting bursts), the Copernicus DEM, the ASF water mask and the S1 precise orbit files — all in parallel. Step 2 runs COMPASS on each burst + date to produce geocoded CSLC HDF5s and static-layer geometry. Step 3 hands everything to dolphin, which writes interferograms, unwrapped phase, a displacement timeseries, and a velocity raster.

In [ ]:
!sweets run {CONFIG_YAML}

## Inspect the outputs

dolphin's output layout under `<work_dir>/dolphin/`:

In [ ]:
!ls {WORK_DIR}/dolphin/

In [ ]:
!ls {WORK_DIR}/dolphin/timeseries/

In [ ]:
# Velocity raster: phase velocity in meters / year (radar wavelength is
# auto-detected from the source — Sentinel-1 at ~5.55 cm here).
!gdalinfo -stats {WORK_DIR}/dolphin/timeseries/velocity.tif | grep -E 'Size|Pixel Size|Unit Type|Minimum|Maximum|StdDev'

In [ ]:
# Plot one of dolphin's wrapped interferograms with the default oil_slick
# cyclic colormap. np.angle maps the complex phase to [-pi, pi], which the
# colormap wraps around so fringes read as contiguous bands.
from sweets.plotting import plot_ifg

ifg_candidates = sorted(
    (WORK_DIR / "dolphin" / "interferograms").rglob("2*.int")
)
assert ifg_candidates, (
    f"no wrapped interferograms found under {WORK_DIR}/dolphin/interferograms/"
)
ifg_path = ifg_candidates[0]
print(f"plotting {ifg_path.relative_to(WORK_DIR)}")

fig, ax = plot_ifg(
    filename=ifg_path,
    figsize=(7, 5),
    title=ifg_path.stem,
    subsample_factor=2,
)

For interactive browsing, open the GeoTIFFs with QGIS, rasterio, rioxarray + hvplot, or any other raster tool. Every file under `dolphin/` is a standard GDAL-readable raster with a real coordinate reference system.